In [2]:
import pandas as pd
import re

df = pd.read_excel("duplicates_report_by_doi.xlsx")

# Keep only non-duplicates (as you asked)
df = df[df["IsDuplicate"] == False].copy()
print("Non-duplicate records:", len(df))

def norm(s):
    return re.sub(r"\s+", " ", str(s).lower()).strip()

# Broader autism block (ASD is ambiguous, so we treat it as "weak" evidence)
AUTISM_STRONG = ["autism", "autistic", "asperger", "pervasive developmental"]
AUTISM_WEAK   = [" asd "]  # handled with padding

# Broader motor block
MOTOR = [
    "gait", "locomot", "walk", "walking", "stride", "step", "foot", "insole",
    "movement", "motor", "kinematic", "biomechan", "posture", "pose", "gesture",
    "stereotyp", "repetitive", "motion", "tracking", "skeleton"
]

# Broader AI block (many papers don’t say “machine learning” explicitly)
AI = [
    "machine learning", "deep learning", "neural", "cnn", "transformer", "vit",
    "classification", "predict", "regression", "model", "algorithm",
    "computer vision", "vision", "pose estimation", "skeleton tracking",
    "graph", "svm", "random forest", "xgboost", "bayesian", "pattern recognition"
]

# Exclusion keywords (high confidence)
ANIMAL = ["mouse", "mice", "rat", "murine", "zebrafish", "primate model"]
NON_ORIGINAL = ["review", "systematic review", "scoping review", "meta-analysis",
                "editorial", "commentary", "letter", "protocol"]
ASD_SPINE = ["adult spinal deformity", "spinal deformity", "spine", "scoliosis"]

def contains_any(text, kws):
    return any(k in text for k in kws)

def score_record(title):
    t = " " + norm(title) + " "  # pad for ASD matching

    # Hard exclusions first
    if contains_any(t, ASD_SPINE) and ("autism" not in t and "autistic" not in t):
        return "Exclude", "ASD refers to adult spinal deformity / spine topic", 0
    if contains_any(t, ANIMAL):
        return "Exclude", "Animal/preclinical study", 0
    if contains_any(t, NON_ORIGINAL):
        return "Exclude", "Not original research (review/editorial/protocol/etc.)", 0

    # Scoring
    score = 0
    reasons = []

    # Autism evidence
    if contains_any(t, AUTISM_STRONG):
        score += 3
        reasons.append("autism-strong")
    elif " asd " in t:
        score += 1
        reasons.append("ASD-weak")

    # Motor evidence
    if contains_any(t, MOTOR):
        score += 2
        reasons.append("motor")

    # AI evidence
    if contains_any(t, AI):
        score += 2
        reasons.append("AI")

    # Decision threshold (tune this!)
    # Recommended for title-only: include if score >= 4
    if score >= 4:
        return "Include", "Score≥4: " + ", ".join(reasons), score

    # “Maybe” bucket to manually check abstract later
    if score == 3:
        return "Maybe", "Borderline (check abstract): " + ", ".join(reasons), score

    return "Exclude", "Insufficient evidence in title (title-only screen)", score

out = df.copy()
out[["Decision", "Reason", "Score"]] = out["Title"].apply(lambda x: pd.Series(score_record(x)))

print("\nCounts:")
print(out["Decision"].value_counts())

out.to_excel("screened_autism_ai_scored.xlsx", index=False)
print("\nSaved: screened_autism_ai_scored.xlsx")

Non-duplicate records: 119

Counts:
Decision
Include    58
Exclude    48
Maybe      13
Name: count, dtype: int64

Saved: screened_autism_ai_scored.xlsx
